# Scattered-point gridding laboratory
Compare nearest-neighbour and Delaunay piecewise-linear interpolation on a synthetic field. Both methods interpolate samples differently and neither extrapolates with a physical model.

In [ ]:
import sys, numpy as np
sys.path.insert(0, '../implementations/python/geophysical-gridding')
from geophysical_gridding import nearest_neighbor_grid, triangulation_grid
rng=np.random.default_rng(12)
points=rng.uniform(-5,5,size=(35,2))
values=np.sin(points[:,0]/2)+np.cos(points[:,1]/3)
axis=np.linspace(-6,6,80)
nearest=nearest_neighbor_grid(points,values,axis,axis,max_distance=1.5)
linear=triangulation_grid(points,values,axis,axis)


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12,5))
for a,z,title in zip(ax,[nearest,linear],['Nearest neighbour','Delaunay linear interpolation']):
    im=a.imshow(z,origin='lower',extent=[axis[0],axis[-1],axis[0],axis[-1]]);a.scatter(points[:,0],points[:,1],c='k',s=8);a.set_title(title);fig.colorbar(im,ax=a)
plt.tight_layout()


## Interpretation
Nearest-neighbour values are constant within Voronoi regions. Triangulation interpolation is piecewise linear and exactly reproduces affine planes. Both are limited to the sampling support; triangulation returns the fill value outside the convex hull. Minimum-curvature gridding and kriging encode different smoothness/covariance assumptions.

## Thin-plate-spline minimum-bending-energy interpolation
This method uses the r² log(r) radial kernel with an affine polynomial tail. Zero smoothing interpolates the samples; positive smoothing relaxes exact fit. It is a thin-plate spline, not the classic Briggs iterative gridding algorithm.

In [ ]:
from geophysical_gridding import thin_plate_spline_grid
thin_plate = thin_plate_spline_grid(points, values, axis, axis, smoothing=0.05)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(thin_plate, origin='lower', extent=[axis[0],axis[-1],axis[0],axis[-1]])
ax.scatter(points[:,0], points[:,1], c='white', edgecolors='black', s=18)
ax.set_title('Thin-plate-spline minimum-bending-energy surface')
fig.colorbar(im, ax=ax); plt.show()


### Compare the assumptions
Nearest neighbour creates Voronoi regions. Delaunay interpolation is piecewise planar and does not extrapolate outside the convex hull. The thin-plate spline is smooth but extrapolates, so inspect its edges and sensitivity to the smoothing parameter. None of these methods supplies a kriging variance or a geological uncertainty model.

## Ordinary kriging with a fixed exponential covariance
Kriging uses an explicit covariance model. Here range, sill and nugget are supplied, not estimated. The returned variance is conditional on those parameters.

In [ ]:
from geophysical_gridding import ordinary_kriging_grid
kriged = ordinary_kriging_grid(points, values, axis, axis, range_=3.0, sill=1.0, nugget=0.05)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, field, title in zip(axes, [kriged['prediction'], kriged['variance']], ['Ordinary kriging prediction', 'Conditional kriging variance']):
    im = ax.imshow(field, origin='lower', extent=[axis[0],axis[-1],axis[0],axis[-1]])
    ax.set_title(title); fig.colorbar(im, ax=ax)
plt.show()


### Variogram assumptions
This is not variogram fitting. The exponential covariance parameters are assumed inputs; different range, sill and nugget values change both predictions and kriging variance. A production analysis should fit and validate a variogram, use spatially blocked validation and account for model uncertainty. The dense solver here is intended for small teaching datasets.